# 01: Exploratory Data Analysis & Data Cleaning Pipeline
### E-commerce Customer Intelligence & Personalization Platform
This notebook explores the raw e-commerce dataset, identifies controlled data anomalies, executes data cleansing, and performs comprehensive customer, product, and transactional EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.ingestion import generate_and_save_dataset
from src.data.cleaning import run_cleaning_pipeline
from src.utils.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
sns.set_theme(style='whitegrid')

## 1. Load Processed Datasets
Inspect the cleaned relational datasets.

In [ ]:
customers_df = pd.read_csv(PROCESSED_DATA_DIR / 'customers.csv')
products_df = pd.read_csv(PROCESSED_DATA_DIR / 'products.csv')
txns_df = pd.read_csv(PROCESSED_DATA_DIR / 'transactions.csv')
reviews_df = pd.read_csv(PROCESSED_DATA_DIR / 'reviews.csv')
interactions_df = pd.read_csv(PROCESSED_DATA_DIR / 'interactions.csv')

print(f'Customers: {len(customers_df):,} | Products: {len(products_df):,} | Txns: {len(txns_df):,} | Reviews: {len(reviews_df):,} | Interactions: {len(interactions_df):,}')

## 2. Customer Demographics & Segment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(customers_df['age'], bins=25, kde=True, ax=axes[0], color='#3b82f6')
axes[0].set_title('Customer Age Distribution')

segment_counts = customers_df['customer_segment'].value_counts()
axes[1].pie(segment_counts, labels=segment_counts.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'))
axes[1].set_title('Customer Segmentation Distribution')
plt.tight_layout()
plt.show()

## 3. Product Catalog & Category Revenue Analysis

In [ ]:
merged_txns = txns_df.merge(products_df[['product_id', 'category']], on='product_id')
cat_rev = merged_txns[merged_txns['return_status'] != 'Returned'].groupby('category')['total_amount'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
cat_rev.plot(kind='bar', color='#10b981')
plt.title('Net Realized Revenue by Product Category ($)')
plt.ylabel('Net Revenue ($)')
plt.xticks(rotation=35)
plt.show()

## 4. Review Ratings & Return Rate Correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=reviews_df, x='rating', palette='Blues_r', ax=axes[0])
axes[0].set_title('Customer Review Rating Distribution (1-5★)')

return_counts = txns_df['return_status'].value_counts()
axes[1].pie(return_counts, labels=return_counts.index, autopct='%1.1f%%', colors=['#10b981', '#ef4444', '#f59e0b'])
axes[1].set_title('Transaction Return Status')
plt.tight_layout()
plt.show()